# 00 · Fuentes históricas y diagnóstico de precios

Antes de construir características o entrenar, comprobamos qué información cubre el desarrollo **1927–1989** y cuánto cambia el análisis según la definición del precio.

Este cuaderno funciona **sin CLI ni archivos de configuración**. No prepara las tablas de los notebooks 01–06 ni las sustituye. Las funciones de `tfm.diagnostics` fijan las fuentes y el periodo de este diagnóstico. Selecciona `.venv`, reinicia el kernel y ejecuta en orden desde la raíz del proyecto o su carpeta `notebooks`.

Solo se analizan importes hasta diciembre de 1989. Los originales completos de Shiller o French pueden contener fechas posteriores; sus lectores excluyen esos importes antes de convertirlos.


## 1. Fuentes y alcance

- **Shiller:** precios medios nominales mensuales del S&P Composite. Conservamos diciembre de 1924–diciembre de 1989 para disponer de historia previa.
- **Yahoo, `^GSPC`:** datos diarios solicitados desde 1920 hasta el inicio de 1990, con fin exclusivo. Calculamos el promedio de los cierres disponibles y el último cierre disponible de cada mes. No utilizamos SPY ni unimos ambos instrumentos.
- **French:** retornos mensuales de doce carteras industriales, ponderadas por valor. Solo el primer bloque mensual del ZIP; no las carteras de pesos iguales ni la tabla anual.
- **FRED BAA y AAA:** tipos de interés de bonos corporativos de dos calidades crediticias. Se mantienen como porcentajes anuales: no son rentabilidades mensuales ni RF. Su diferencia se expresa en puntos porcentuales.

Las carteras de French son industrias, no ETF ni acciones individuales. Su universo no se limita a los componentes del S&P 500; describen un mercado estadounidense más amplio. Los originales actuales reconstruyen historia; disponer de una observación no acredita por sí solo su publicación ni su versión en cada fecha.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from tfm.diagnostics import (
    START, END, PRICE_NAMES, SOURCE_URLS, load_diagnostic_sources,
    coverage_table, compare_price_definitions, feature_availability,
)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise ValueError("Abre el notebook desde la raíz del proyecto o su carpeta notebooks.")
print("Python:", sys.executable)
print("Proyecto:", ROOT)
print(f"Desarrollo: {START} → {END}. No se entrenan modelos ni se evalúa desde 1990.")
display(pd.DataFrame([{"Fuente": name, "Enlace": url} for name, url in SOURCE_URLS.items()]))


## 2. Leer o descargar originales y comprobar cobertura

**Esta celda puede descargar y guardar originales.** Si existe `data/raw/shiller.xls`, solo lo lee. Las nuevas fuentes se conservan por separado en `data/raw/diagnosis/`: ZIP industrial, CSV originales de FRED y un Parquet de la respuesta diaria de Yahoo. Si existen, los reutiliza; no actualiza ni sustituye automáticamente las copias.

Los lectores rechazan meses interiores ausentes o duplicados y valores inválidos. La tabla informa también de los meses iniciales de desarrollo que una fuente no cubre. No los rellena.

La cobertura mostrada es la **analizada en este diagnóstico**, con fechas solicitadas o seleccionadas, no necesariamente el principio y final absolutos de la fuente. Tener un dato en un mes no certifica todas las sesiones bursátiles ni que el último dato sea el cierre oficial del mes.


In [ ]:
print("Leyendo originales; la primera descarga de las fuentes puede tardar…", flush=True)
fuentes = load_diagnostic_sources(ROOT)
cobertura = coverage_table(fuentes)
vista_cobertura = cobertura.drop(columns="source_url").rename(columns={
    "source": "Fuente", "first_analyzed_month": "Primer mes analizado", "last_analyzed_month": "Último mes analizado",
    "rows": "Filas mensuales", "development_months": "Meses de desarrollo", "missing_development_months": "Meses de desarrollo ausentes",
    "first_missing_months": "Primeros meses ausentes",
})
display(vista_cobertura.set_index("Fuente"))
print("Primer mes de Yahoo:")
display(fuentes["yahoo"].head(1))


## 3. Comparar tres definiciones del precio

Usamos tres series separadas:

1. **Media de Shiller frente a media de Yahoo:** ayuda a detectar diferencias de proveedor, reconstrucción o sesiones disponibles.
2. **Media de Yahoo frente a último cierre de Yahoo:** mantiene un mismo proveedor y permite estudiar el efecto de promediar.
3. **Media de Shiller frente a último cierre de Yahoo:** muestra la diferencia conjunta que encontraríamos al cambiar de base.

Se excluye la media del primer mes de Yahoo, porque su cobertura inicial puede ser parcial. Se toma el intervalo continuo común y se utiliza la misma historia inicial para todas las series. Los indicadores se calculan con las fórmulas originales: momentum `P[m−1]/P[m−12]−1`, reversión `P[m]/P[m−1]−1`, señal para m+1. No se utiliza un cierre como si fuera un promedio de Shiller.

La comparación de señales empieza después del calentamiento y termina con meses objetivo hasta diciembre de 1989. No debe confundirse ese intervalo reducido con todo el desarrollo del TFM.


In [ ]:
contraste = compare_price_definitions(fuentes["shiller"], fuentes["yahoo"])
precios = contraste["prices"]
senales = contraste["signals"]
metricas = contraste["statistics"]
acuerdos = contraste["agreement"]
print(f"Precios comunes: {precios.period.iloc[0]} → {precios.period.iloc[-1]} ({len(precios)} meses)")
print(f"Meses objetivo comunes con ambas reglas: {metricas.first_target.iloc[0]} → {metricas.last_target.iloc[0]} ({metricas.months.iloc[0]} meses)")
vista = metricas.set_index("basis")[["annualized_price_volatility", "lag1_price_return_correlation",
                                        "momentum_mean_gross_price_payoff", "reversal_mean_gross_price_payoff"]].copy()
for column in ["annualized_price_volatility", "momentum_mean_gross_price_payoff", "reversal_mean_gross_price_payoff"]:
    vista[column] *= 100
vista = vista.rename(index=PRICE_NAMES, columns={
    "annualized_price_volatility": "Volatilidad anualizada por precio (%)",
    "lag1_price_return_correlation": "Correlación con el mes anterior",
    "momentum_mean_gross_price_payoff": "Resultado medio bruto momentum (% mensual)",
    "reversal_mean_gross_price_payoff": "Resultado medio bruto reversión (% mensual)",
})
display(vista.round(4))
vista_acuerdos = pd.DataFrame({
    "Comparación": acuerdos.left.map(PRICE_NAMES) + " / " + acuerdos.right.map(PRICE_NAMES),
    "Regla": acuerdos.rule, "Meses": acuerdos.months, "Señales distintas": acuerdos.different_signals,
    "Señales distintas (%)": 100 * acuerdos.different_fraction,
})
display(vista_acuerdos.round(3))

print("Mayores diferencias entre las medias: inspección descriptiva, sin corregir valores")
discrepancias = contraste["source_discrepancies"].copy()
for column in ["shiller_minus_yahoo_mean_fraction", "shiller_minus_yahoo_close_fraction"]:
    discrepancias[column] *= 100
display(discrepancias.rename(columns={
    "period": "Mes", "shiller_mean": "Shiller", "yahoo_mean": "Media Yahoo", "yahoo_close": "Último cierre Yahoo",
    "shiller_minus_yahoo_mean_fraction": "Shiller / media Yahoo − 1 (%)",
    "shiller_minus_yahoo_close_fraction": "Shiller / cierre Yahoo − 1 (%)",
}).round(4))


## 4. Qué significan las medidas

La volatilidad resume las oscilaciones de las variaciones de precio, anualizada con raíz de doce. La correlación con el mes anterior describe si movimientos consecutivos tienden a ir en la misma dirección o en la contraria; no demuestra predictibilidad útil.

Promediar puede introducir dependencia entre variaciones que no estaba en la serie original. Esto es especialmente relevante para reversión. Véase la discusión de agregación temporal en [Crawley, Reserva Federal (2019)](https://www.federalreserve.gov/econres/feds/in-search-of-lost-time-aggregation.htm), que parte del resultado de Working (1960). La aplicación a nuestro diagnóstico es una motivación, no una demostración de que toda diferencia tenga esa causa.

El resultado bruto de una señal se calcula como `señal decidida en m × variación de precio en m+1`. La media incluye los meses en efectivo con contribución de precio cero. **No incorpora dividendos, RF, préstamo, comisiones, reinversión ni una cuenta de capital.** Sirve para estudiar cómo cambia la relación entre señal y movimiento posterior; no sustituye las simulaciones financieras de los otros cuadernos.

Los porcentajes de señales distintas miden desacuerdo, no qué fuente tiene razón. Las fuentes se valorarán por su definición, cobertura y coherencia, no por producir la mayor ganancia histórica.

La tabla de mayores diferencias ayuda a localizar observaciones concretas. En la revisión inicial destacó julio de 1974: Shiller mostraba 79,31 frente a una media de Yahoo de aproximadamente 82,8191, mientras su último cierre era aproximadamente 79,31. La coincidencia con el cierre no acredita por sí sola la causa. El XLS publicado consultado el 1 de noviembre de 2026 también contiene 79,31. El dato se conserva y se registra la discrepancia, sin corregirlo automáticamente.


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
indices = precios.set_index("period")
fechas = pd.PeriodIndex(indices.index, freq="M").to_timestamp(how="end")
for name in PRICE_NAMES:
    ejes[0].plot(fechas, 100 * (indices[name] / indices[name].iloc[0] - 1), label=PRICE_NAMES[name], alpha=0.8)
ejes[0].set(title="Variación acumulada de cada serie de precios", xlabel="Año", ylabel="Variación desde el primer mes común (%)")
ejes[0].legend(fontsize=8)
ejes[0].grid(alpha=0.25)
diferencia = 100 * (indices.shiller_mean / indices.yahoo_mean - 1)
ejes[1].plot(fechas, diferencia, color="#204d74")
ejes[1].axhline(0, color="black", linewidth=0.8)
ejes[1].set(title="Diferencia relativa entre las dos medias", xlabel="Año", ylabel="Shiller / media Yahoo − 1 (%)")
ejes[1].grid(alpha=0.25)
plt.show()


## 5. Disponibilidad inicial de características propuestas

Este bloque cuenta meses utilizables; **no selecciona variables por rendimiento ni construye todavía el conjunto final de entrenamiento**.

Para las características de precio se utiliza información hasta el mes de decisión. Para crédito e industrias se propone inicialmente **un mes adicional de rezago** por su publicación posterior al periodo observado. Es un supuesto conservador que deberá revisarse con la documentación de disponibilidad; no acredita versiones históricas ni elimina todas las revisiones.

La amplitud propuesta utiliza índices de riqueza construidos con los retornos industriales y una media de doce observaciones. La dispersión es entre industrias, no entre acciones individuales. El calentamiento y el rezago pueden dejar meses iniciales sin estas variables: no se rellenan como si hubieran existido.

VIX y ETF sectoriales no figuran como candidatos del modelo principal porque no cubren el entrenamiento 1927–1989. Los tipos oficiales a diez años y la pendiente 10–2 tampoco cubren todo ese periodo. El diferencial Baa–Aaa compara calidades crediticias; no es idéntico a un diferencial corporativo frente al Tesoro.


In [ ]:
disponibilidad = feature_availability(fuentes)
display(disponibilidad.rename(columns={
    "feature": "Característica propuesta", "first_available_target": "Primer objetivo bajo el rezago propuesto",
    "last_available_target": "Último mes objetivo disponible", "development_months": "Meses de desarrollo",
    "extra_publication_lag_months": "Rezago adicional propuesto (meses)",
}).set_index("Característica propuesta"))
print("La cobertura temporal no demuestra capacidad predictiva ni disponibilidad histórica exacta.")


## 6. Guardar las tablas del diagnóstico

**Esta celda escribe únicamente en `outputs/diagnosis/`.** Sustituye las tablas de este diagnóstico con nombres fijos; no modifica `data/processed/` ni los resultados de los notebooks 01–06. Los resultados guardados no contienen observaciones de prueba desde 1990.


In [ ]:
salida = ROOT / "outputs/diagnosis"
salida.mkdir(parents=True, exist_ok=True)
tablas = {"coverage": cobertura, "feature_availability": disponibilidad, **contraste}
for nombre, tabla in tablas.items():
    tabla.to_parquet(salida / f"{nombre}.parquet", index=False)
print("Tablas guardadas en:", salida)


## 7. Decisiones que permite preparar

- Mantener identificadas las medias y los cierres y reconocer sus diferencias.
- Documentar qué fuentes y características cubren el desarrollo y qué meses se pierden por calentamiento y rezagos.
- Conservar Shiller como base principal mientras se cierra el diagnóstico metodológico, sin empalmar con cierres.
- Precisar los límites de la aproximación, especialmente en reversión, antes de entrenar.

No se han modificado las reglas, elegido hiperparámetros ni evaluado un meta-modelo. Quedan por completar el análisis de disponibilidad histórica, las características definitivas, las etiquetas y la validación. Los notebooks 01–06 también se ejecutan directamente, sin CLI ni ficheros de configuración.

Antes de guardar para Git, limpia las salidas y los contadores. Los originales y las tablas de datos permanecen fuera de Git.
